# L06. Backpropagation Basics

---

## 학습 목표

1. `loss.backward()`가 하는 일을 설명할 수 있다.
2. 각 층의 weight와 bias에 gradient가 어떻게 저장되는지 확인할 수 있다.
3. MNIST 데이터 한 배치에 대해 backward pass를 수행할 수 있다.
4. gradient를 이용해 파라미터를 직접 한 번 업데이트할 수 있다.
5. update 전후 loss 변화를 비교할 수 있다.


## 1. Backpropagation 개요

신경망 학습은 보통 다음 순서로 진행됩니다.

```
forward -> loss 계산 -> backward -> update
```

- **forward**: 입력을 넣어 출력값을 계산
- **loss**: 예측과 정답의 차이를 계산
- **backward**: 각 파라미터의 gradient 계산
- **update**: gradient를 사용해 파라미터 수정

이 노트북에서는 이 흐름을 작은 예제로 확인합니다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

np.random.seed(42)
torch.manual_seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('NumPy version  :', np.__version__)
print('PyTorch version:', torch.__version__)


## 2. 작은 식에서 gradient 보기

다음 식을 생각해봅시다.

$$
y = wx + b, \quad L = (y - t)^2
$$

PyTorch는 이 계산 그래프를 추적하고, `backward()`를 호출하면 필요한 미분값을 자동으로 계산합니다.


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(-0.4, requires_grad=True)
b = torch.tensor(0.3, requires_grad=True)
t = torch.tensor(1.0)

y = w * x + b
loss = (y - t) ** 2
loss.backward()

print(f'y        = {y.item():.4f}')
print(f'loss     = {loss.item():.4f}')
print(f'dL/dw    = {w.grad.item():.4f}')
print(f'dL/db    = {b.grad.item():.4f}')
print(f'dL/dx    = {x.grad.item():.4f}')


## 3. MNIST 데이터 확인

이번 실습에서는 `MNIST`를 사용합니다.

| 항목 | 내용 |
|---|---|
| 이미지 크기 | `28 x 28` |
| 채널 수 | 1 (grayscale) |
| 클래스 수 | 10 |
| 예시 | 손글씨 숫자 0~9 |

먼저 데이터를 불러오고, 이미지와 label 분포를 확인합니다.


In [ ]:
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
subset = Subset(train_dataset, range(512))
train_loader = DataLoader(subset, batch_size=64, shuffle=True)

images, labels = next(iter(train_loader))
print('images shape:', tuple(images.shape))
print('labels shape:', tuple(labels.shape))
print('first 10 labels:', labels[:10].tolist())


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(10, 3))
axes = axes.ravel()
for i, ax in enumerate(axes):
    ax.imshow(images[i, 0], cmap='gray')
    ax.set_title(str(labels[i].item()))
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
label_counts = torch.bincount(labels, minlength=10)
plt.figure(figsize=(6, 3))
plt.bar(range(10), label_counts.numpy(), color='tab:blue')
plt.xticks(range(10))
plt.xlabel('Digit class')
plt.ylabel('Count in batch')
plt.title('Label Distribution of One Batch')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 4. 모델 만들기

이번에는 가장 단순한 DNN을 사용합니다.

```
Flatten -> Linear -> ReLU -> Linear
```

출력층의 결과는 10개 클래스에 대한 score(logit)입니다.


In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
criterion = nn.CrossEntropyLoss()

print(model)


## 5. Forward와 Loss 계산

입력 배치를 모델에 넣고, `CrossEntropyLoss`를 계산합니다.


In [ ]:
logits = model(images)
loss = criterion(logits, labels)

print('logits shape:', tuple(logits.shape))
print(f'loss = {loss.item():.4f}')


## 6. Backward 수행하기

이제 `loss.backward()`를 호출해 gradient를 계산합니다.

계산된 gradient는 각 파라미터의 `.grad` 속성에 저장됩니다.


In [ ]:
model.zero_grad()
loss.backward()

for name, param in model.named_parameters():
    print(name)
    print('  shape     :', tuple(param.shape))
    print('  grad shape:', tuple(param.grad.shape))
    print('  grad norm :', f'{param.grad.norm().item():.6f}')


## 7. Gradient를 직접 사용해 update 하기

가장 단순한 update는 다음과 같습니다.

$$
parameter \leftarrow parameter - learning\ rate \times gradient
$$

여기서는 optimizer를 쓰지 않고 직접 한 번 업데이트합니다.


In [ ]:
lr = 0.1

with torch.no_grad():
    before_loss = criterion(model(images), labels).item()

    for param in model.parameters():
        param -= lr * param.grad

    after_loss = criterion(model(images), labels).item()

print(f'before update loss = {before_loss:.4f}')
print(f'after update loss  = {after_loss:.4f}')


## 8. 여러 번 반복해 보기

같은 배치에 대해 여러 번 update 하면서 loss와 accuracy 변화를 확인합니다.


In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
criterion = nn.CrossEntropyLoss()
lr = 0.1
loss_history = []
acc_history = []

for step in range(30):
    logits = model(images)
    loss = criterion(logits, labels)

    model.zero_grad()
    loss.backward()

    with torch.no_grad():
        for param in model.parameters():
            param -= lr * param.grad

    preds = logits.argmax(dim=1)
    acc = (preds == labels).float().mean().item()
    loss_history.append(loss.item())
    acc_history.append(acc)

print(f'initial loss = {loss_history[0]:.4f}')
print(f'final loss   = {loss_history[-1]:.4f}')
print(f'initial acc  = {acc_history[0]:.4f}')
print(f'final acc    = {acc_history[-1]:.4f}')


In [ ]:
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.plot(loss_history, marker='o')
plt.title('Loss over updates')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(acc_history, marker='o', color='tab:orange')
plt.title('Accuracy over updates')
plt.xlabel('Step')
plt.ylabel('Accuracy')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 9. 핵심 정리

- `loss.backward()`는 loss를 기준으로 gradient를 계산한다.
- 계산된 gradient는 각 파라미터의 `.grad`에 저장된다.
- gradient를 이용하면 파라미터를 직접 업데이트할 수 있다.
- update를 반복하면 loss가 감소하는 모습을 확인할 수 있다.

→ 다음 토픽: **Optimization & Regularization**


## 10. 연습 문제

### 📝 Exercise 1

`w`, `b`, `x`의 gradient 중 각각이 무엇을 의미하는지 설명해보세요.

### 📝 Exercise 2

learning rate를 `0.01`, `0.5`로 바꿔서 loss 변화를 비교해보세요.

- 어느 경우가 더 천천히 줄어드는가?
- 어느 경우가 너무 크게 움직일 수 있는가?

### 📝 Exercise 3

모델의 hidden dimension을 `128` 대신 `32`, `256`으로 바꿔보세요.

- gradient shape는 어떻게 달라지는가?
- loss 감소 속도는 어떻게 달라지는가?

### 📝 Exercise 4

반복 횟수를 `30`에서 `100`으로 늘려서 같은 배치에서 accuracy가 어떻게 변하는지 확인해보세요.

### 📝 Exercise 5

`ReLU`를 `Sigmoid`로 바꿔서 다시 실행해보세요.

- loss 감소 패턴이 어떻게 달라지는가?
- gradient norm은 어떻게 달라지는가?
